# pMoE and MGE: comparing head settings (v2)

**What it does.** Takes a **frozen, pretrained backbone** and trains the two most interesting heads on it, **pMoE** (experts that look at a few image patches) and **MGE** (one expert per backbone depth), each with several settings (the two grids below), so you can see which setting works best.

**How to run**
1. In *Choose the backbone*, write the path of a saved backbone in `BACKBONE_PATH` (for example one from `MoE_comparisons/models/`), or leave it `None` to train a new one (saved under an automatic name in `pMoE_MGE_finetune_v2/models/`). This notebook has its own backbone code and does not need any other notebook.
2. Run the cells from top to bottom. Every head run trains only the head, and the backbone is never changed.
3. Add, remove or change entries of the two grids as you like. To retrain just one setting, leave only that entry in its grid and run again: the comparison at the end also reads the logs of earlier runs on the same backbone from `pMoE_MGE_finetune_v2/history_logs/`.
4. Logs and head weights go to this notebook's own folder, `pMoE_MGE_finetune_v2/`.

Only the **validation** split is used to choose settings; the test split is used once at the end, for the two winners.

In [1]:
import os, sys, subprocess
import warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')   # take GPU memory as needed, not all of it at start (kernels can then share a GPU)

REPO_URL = 'https://github.com/durak-dev/ImageClassification.git'
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Colab starts with an empty VM: clone the repo (code AND data live in it).
    # On later runs the folder already exists, so just pull the newest commits.
    ROOT = '/content/ImageClassification'
    if not os.path.exists(ROOT):
        subprocess.run(['git', 'clone', REPO_URL, ROOT], check=True)
    else:
        # fetch + hard reset instead of `pull`: always lands on the newest origin commit, even if the VM copy diverged
        subprocess.run(['git', '-C', ROOT, 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', ROOT, 'reset', '--hard', '@{u}'], check=True)
else:
    # Local: the notebook sits in the project root, but also work if it was
    # started from a subfolder.
    cwd = os.path.abspath(os.getcwd())
    ROOT = cwd if os.path.isdir(os.path.join(cwd, 'utils')) else os.path.dirname(cwd)

os.chdir(ROOT)
sys.path.insert(0, ROOT)          # so `import utils.funcs` finds OUR utils folder

DATA_DIR = os.path.join(ROOT, 'data')
ORIG_DIR = os.path.join(DATA_DIR, 'original')            # raw images/ + labels/ (placed manually, never modified)
MANIFEST_PATH = os.path.join(DATA_DIR, 'split_manifest.csv')   # which file belongs to which split
SEED = 42
NB_DIR = 'pMoE_MGE_finetune_v2'   # this notebook's output folder (relative to the project root)
MODELS_DIR = f'{NB_DIR}/models'          # backbones and head weights
HISTORY_DIR = f'{NB_DIR}/history_logs'   # one CSV log per training run
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(HISTORY_DIR, exist_ok=True)
print('Running on:', 'Google Colab' if IN_COLAB else 'local machine')
print('Project root:', ROOT)

Running on: local machine
Project root: /workspace/ImageClassification


In [ ]:
# the kernel survives between runs: drop every cached utils module so the code git just pulled is what gets imported
for _m in [m for m in sys.modules if m == 'utils' or m.startswith('utils.')]:
    del sys.modules[_m]
from utils.funcs import *
from utils import config, github_sync, plotting, resnet
from utils.github_sync import save_csv, load_csv, push_files
from utils.MoE import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import f1_score

### Where the results go
Logs go to `pMoE_MGE_finetune_v2/history_logs/`, backbones and model weights to `pMoE_MGE_finetune_v2/models/` (this notebook's own output folder). A RunPod pod or Colab VM loses its disk when it ends, so the next cell offers two ways to keep them:
* **Push to GitHub:** paste a GitHub token into the box (hidden, kept in memory only). Everything is then committed to the `checkpoints` branch as soon as it is written, and a new pod downloads the saved backbone from there instead of training it again. Leave the box empty to skip.
* **Copy to your PC:** run `.\pull_results.ps1 -Ip <ip> -Port <port>` before you terminate the pod (see the README).

Running locally you can ignore this cell: everything stays in the project folder.

In [3]:
# Is the disk temporary? Colab VM, or a RunPod pod (pod_bootstrap.sh clones the repo into /workspace).
ON_POD = ROOT.startswith('/workspace/') or bool(os.environ.get('RUNPOD_POD_ID'))
REMOTE = IN_COLAB or ON_POD
if REMOTE:
    config.ensure_github()      # asks once for a GitHub token (hidden box, kept in memory only); empty = no upload
USE_GITHUB = REMOTE and config.github_ready()

if USE_GITHUB:
    _, repo, branch = config.github_settings()
    print(f'Results -> GitHub {repo}@{branch}: every log, backbone and weight file is committed when it is written')
elif ON_POD:
    print('Results stay on the pod. Copy them to your PC before terminating it:  .\\pull_results.ps1 -Ip <ip> -Port <port>')
else:
    print(f'Results -> local folders ({HISTORY_DIR}/, {MODELS_DIR}/)')

Results -> GitHub durak-dev/ImageClassification@checkpoints: every log, backbone and weight file is committed when it is written


In [3]:
# Recreate data/{split}/ with the corrected images + labels if they are missing (fresh Colab VM).
# Needs split_manifest.csv, data/original/ and the two checkpoint files from the cloned repo.
if not all(os.path.exists(os.path.join(split_dir(DATA_DIR, s, 'labels'), CORRECTED_LABELS_FILE)) for s in SPLITS):
    materialize_splits(ORIG_DIR, DATA_DIR, pd.read_csv(MANIFEST_PATH))
    rebuild_corrected(pd.read_csv(os.path.join(ROOT, 'checkpoints', 'df_fixed.csv')), DATA_DIR,
                      labels_corrected=pd.read_csv(os.path.join(ROOT, 'checkpoints', 'labels_corrected.csv')))

In [4]:
dfs = {s: pd.read_csv(os.path.join(split_dir(DATA_DIR, s, "labels"), CORRECTED_LABELS_FILE)) for s in ["train", "val", "test"]}
classes = sorted(dfs["train"]["class_id"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}

## Data
Images are resized to 64x64. **Only the training set is rebalanced** (classes with few images are repeated up to the median class size); validation and test stay as they are.

In [5]:
SIZE, BATCH, AUTOTUNE = 64, 128, tf.data.AUTOTUNE
CLASS_BALANCE = 'oversample'         # train set only: None | "oversample" | "undersample"  (see funcs.balance_frame)
CLASS_BALANCE_TARGET = 'median'  # images per class: None (max / min) | "min" | "median" | "max" | int

def load(path, y):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.image.resize(img, (SIZE, SIZE)), y

def make_ds(split, shuffle):
    df = dfs[split]
    if split == "train" and CLASS_BALANCE:      # never resample val / test
        df = balance_frame(df, CLASS_BALANCE, CLASS_BALANCE_TARGET, seed=SEED)
        n = df["class_id"].value_counts()
        print(f"train balanced ({CLASS_BALANCE}): {len(df)} images, {n.min()}-{n.max()} per class")
    paths = (f"{DATA_DIR}/{split}/images_corrected/" + df["filename"].astype(str)).values
    labels = df["class_id"].map(class_to_idx)
    assert labels.notna().all(), f"{split} has labels not present in train"
    ds = tf.data.Dataset.from_tensor_slices((paths, labels.values.astype("int32")))
    if shuffle:
        ds = ds.shuffle(len(df), seed=SEED)
    return ds.map(load, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

train, val, test = make_ds("train", True), make_ds("val", False), make_ds("test", False)
print(len(classes), "classes")
print(dfs["train"]["filename"].iloc[0], os.path.exists(f"{DATA_DIR}/train/images_corrected/" + str(dfs["train"]["filename"].iloc[0])))
for x, y in train.take(1):
    print(x.shape, x.dtype, float(tf.reduce_min(x)), float(tf.reduce_max(x)))

train balanced (oversample): 38600 images, 193-193 per class


I0000 00:00:1795030538.797751  120227 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 867 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe, pci bus id: 0000:00:0c.0, compute capability: 8.0


200 classes
image_000002.JPEG True
(128, 64, 64, 3) <dtype: 'float32'> 0.0 255.0


## Settings
Everything you might want to change is in the next cell and in the two grids further down.

* **Backbone** (only used when `BACKBONE_PATH = None`) - `drop` is the dropout inside the backbone and `weight_decay` the AdamW weight decay.
* **Heads** - every model trains only its own head on top of the frozen backbone.

In [ ]:
# ======== Settings ========

# --- backbone training (only used when BACKBONE_PATH = None)
BACKBONE = dict(
    drop=0.2,           # dropout inside the backbone (after stages 2 and 3)
    head_drop=0.3,      # dropout of the linear head used only for pretraining
    epochs=30,          # pretraining epochs (early stopping usually stops earlier)
    lr=1e-3,            # pretraining learning rate
    weight_decay=1e-4,  # AdamW weight decay
)

# --- heads: every model trains its own head on the frozen backbone
EPOCHS = 40               # head training epochs (early stopping, patience 8)
LR = 1e-3                 # head learning rate
USE_AUG = True            # random flip + shift for every model
FEATURE_DROP = 0.3        # CNN / MoE: dropout on the 256 pooled features
N_EXPERTS = 4             # MoE / pMoE: number of experts
BALANCE = 0.05            # MoE: weight of the load-balancing loss
L_PATCHES = 8             # pMoE: patches each expert keeps (at most 64)
DIVERSITY_WEIGHT = 0.1    # MGE: weight of the diversity penalty

aug = tf.keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomTranslation(0.1, 0.1)])
AUG = aug if USE_AUG else None


def run(model, name, epochs=EPOCHS, gate=False):
    """Train with the default callbacks (val macro-F1, CSV log, early stopping) and save <HISTORY_DIR>/<name>.csv."""
    cbs, mon = default_callbacks(val, name=name, log_dir=HISTORY_DIR, gate=gate)
    history = model.fit(train, validation_data=val, epochs=epochs, callbacks=cbs)
    if USE_GITHUB:
        push_files({f'{HISTORY_DIR}/{name}.csv': os.path.join(HISTORY_DIR, f'{name}.csv')}, f'history log: {name}')
    plot_history(history)
    return history, mon

### Run bookkeeping

In [ ]:
import gc
BEST_F1 = {}   # model type -> best val macro-F1 so far (only that run keeps its Keras model)
RUNS = {}   # run name -> dict(model, settings, log (DataFrame), net (Keras model; only kept for the best run of each type, else None))


def get_run(name, build, model, settings):
    """Train build() and register the run in RUNS (its log is also saved in HISTORY_DIR)."""
    print(f'===== {name}')
    tf.keras.backend.clear_session()
    net = build()
    log = pd.DataFrame(run(net, name)[0].history)
    # GPU memory: keep only the best model of each type (the winners cells need just those), free the rest
    f1 = log['val_f1'].max()
    if f1 > BEST_F1.get(model, -1):
        BEST_F1[model] = f1
        for r in RUNS.values():
            if r['model'] == model:
                r['net'] = None
    else:
        net = None
    gc.collect()
    tf.keras.backend.clear_session()
    RUNS[name] = dict(model=model, settings=settings, log=log, net=net)
    return log, net

## The backbone
A ResNet written for 64x64 images (no early downsampling), written as the function `backbone` in the first code cell below, so you can change it right there (layers, widths, number of blocks). It must return the three feature maps `[s1, s2, s3]` (shallow to deep): `s1` 32x32x64, `s2` 16x16x128, `s3` 8x8x256. pMoE reads `s3`, MGE reads all three.

A saved backbone is a complete `.keras` model (layers and weights), so it can be loaded without this code: editing `backbone` only matters when you train a **new** backbone, and old files keep the layers they were trained with. The cell after the code defines `train_backbone()`; the one after that chooses the backbone.

In [ ]:
# The backbone: edit it freely. It must return [s1, s2, s3] (shallow -> deep); the settings come from BACKBONE above.
# Only used when training a NEW backbone: a saved backbone carries its own layers, so editing this cell does not change it.
# (It replaces the default `backbone` of utils/MoE.py, which is not used here.)
def backbone(inp):
    """Conv layers only (no augmentation) -> [s1, s2, s3], shallow -> deep."""
    drop = BACKBONE['drop']
    x = layers.Rescaling(1 / 255)(inp)
    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)     # stem
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # stage 1: 64 x 64 x 32, no downsampling
    for _ in range(3):
        x = res_block(x, 32, downsample=False)

    # stage 2: 32 x 32 x 64
    x = res_block(x, 64)
    for _ in range(5):
        x = res_block(x, 64, downsample=False)
    s1 = x                                        # MGE expert 0
    x = layers.SpatialDropout2D(drop)(x)

    # stage 3: 16 x 16 x 128
    x = res_block(x, 128)
    for _ in range(7):
        x = res_block(x, 128, downsample=False)
    s2 = x                                        # MGE expert 1
    x = layers.SpatialDropout2D(drop)(x)

    # stage 4: 8 x 8 x 256
    x = res_block(x, 256)
    for _ in range(2):
        x = res_block(x, 256, downsample=False)
    s3 = x                                        # MGE expert 2, pMoE, CNN / MoE (pooled)

    return [s1, s2, s3]

In [ ]:
# Train a NEW backbone (only runs when BACKBONE_PATH is None in the cell below).
# It is trained with a plain linear head, then saved as a complete .keras file (layers + weights) in MODELS_DIR under an
# automatic, unique name. A file is never overwritten.
def train_backbone():
    tf.keras.backend.clear_session()
    trunk = resnet.make_trunk(SIZE, backbone)
    path = resnet.new_trunk_path(MODELS_DIR, trunk)
    inp = tf.keras.Input((SIZE, SIZE, 3))
    x = AUG(inp) if AUG is not None else inp
    x = layers.GlobalAveragePooling2D()(trunk(x)[-1])
    x = layers.Dropout(BACKBONE['head_drop'])(x)
    pre = tf.keras.Model(inp, layers.Dense(len(classes))(x), name='pretrain')
    pre.compile(tf.keras.optimizers.AdamW(BACKBONE['lr'], weight_decay=BACKBONE['weight_decay']),
                tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
    run(pre, resnet.trunk_name(path) + '_pretrain', epochs=BACKBONE['epochs'])
    resnet.save_trunk(trunk, path)                  # best epoch: EarlyStopping restored the best weights
    print(f'saved {os.path.getsize(path) / 1e6:.1f} MB -> {path}')
    if USE_GITHUB:
        push_files({path: path}, f'backbone: {path}')
    tf.keras.backend.clear_session()
    return path


## Choose the backbone
`BACKBONE_PATH` decides which backbone every head below is trained on, and the backbone is **frozen**: no run changes it, so every head sees exactly the same features.

* `BACKBONE_PATH = 'pMoE_MGE_finetune_v2/models/<file>.keras'` (path relative to the project folder) loads that backbone. It can come from either notebook.
* `BACKBONE_PATH = None` trains a new backbone first (the code above), saves it in this notebook's `models/` folder under an automatic name and prints that name, so you can paste it here next time. Files are never overwritten.

The file name goes into the log names, so runs on different backbones never overwrite each other.

In [ ]:
# The backbone EVERY head below is trained on. It is FROZEN: no run changes it.
#   path -> load that backbone, e.g. 'pMoE_MGE_finetune_v2/models/backbone_6072k_20251118-210000.keras'
#   None -> train a NEW backbone now (saved in pMoE_MGE_finetune_v2/models/ under an automatic name, printed below)
BACKBONE_PATH = None

if BACKBONE_PATH is None:
    BACKBONE_PATH = train_backbone()
    print(f'To reuse it later, write  BACKBONE_PATH = {BACKBONE_PATH!r}')

resnet.fetch_if_missing(BACKBONE_PATH, USE_GITHUB)         # a new pod: download it from GitHub
BACKBONE_ID = resnet.trunk_name(BACKBONE_PATH)             # part of every log name
assert resnet.trunk_size(BACKBONE_PATH) == SIZE, f'{BACKBONE_PATH} was trained on other image sizes than SIZE = {SIZE}'

BB = resnet.make_frozen_backbones(BACKBONE_PATH, feature_drop=FEATURE_DROP)
_inp = tf.keras.Input((SIZE, SIZE, 3))
print('using the frozen backbone', BACKBONE_PATH)
print('feature maps (H, W, C):', [tuple(m.shape[1:]) for m in BB.maps(_inp)])
del _inp
tf.keras.backend.clear_session()


## Head settings to compare
Each entry is one run; its key is the short name in the logs. The first entry of each grid uses the settings above (`N_EXPERTS`, `L_PATCHES`, `DIVERSITY_WEIGHT`). Add, remove or change entries freely; to retrain just one setting, keep only that entry.

In [ ]:
# pMoE: number of experts and patches kept per expert.   MGE: weight of the diversity penalty.
PMOE_GRID = {f'e{n}_l{l}': dict(n_experts=n, l=l) for n, l in [(N_EXPERTS, L_PATCHES), (8, 8), (8, 16)]}
MGE_GRID = {f'div{w}': dict(diversity_weight=w) for w in [DIVERSITY_WEIGHT, 0, 0.3]}
print(list(PMOE_GRID), list(MGE_GRID))

## pMoE sweep

In [ ]:
for key, cfg in PMOE_GRID.items():
    get_run(f'pmoe_{key}_{BACKBONE_ID}',
            lambda: build_pmoe(SIZE, len(classes), augmentation=AUG, backbone_fn=BB.last, lr=LR, **cfg),
            'pmoe', key)

## MGE sweep

In [ ]:
for key, cfg in MGE_GRID.items():
    get_run(f'mge_{key}_{BACKBONE_ID}',
            lambda: build_mge(SIZE, len(classes), augmentation=AUG, backbone_fn=BB.maps, lr=LR, **cfg),
            'mge', key)

## Comparison
All choices are made on the **validation** set. For MGE the figures use its `ensemble` output.
Logs of earlier runs on the same backbone are added, so retraining a single setting still shows everything.

In [ ]:
# add the runs on THIS backbone whose logs already exist (earlier sessions, other grids)
for model in ('pmoe', 'mge'):
    for f in sorted(os.listdir(HISTORY_DIR)):
        if f.startswith(f'{model}_') and f.endswith(f'_{BACKBONE_ID}.csv'):
            name = f[:-4]
            if name not in RUNS:
                RUNS[name] = dict(model=model, settings=name[len(model) + 1:-len(BACKBONE_ID) - 1],
                                  log=pd.read_csv(os.path.join(HISTORY_DIR, f)), net=None)


def runs_of(model):
    return {n: r for n, r in RUNS.items() if r['model'] == model}


def label(r):
    return f"{r['model']} {r['settings']}"


def val_acc(log):
    return log['val_ensemble_accuracy'] if 'val_ensemble_accuracy' in log else log['val_accuracy']


summary = pd.DataFrame([dict(run=n, model=r['model'], settings=r['settings'],
                             epochs_run=len(r['log']), best_val_f1=r['log']['val_f1'].max(),
                             at_epoch=int(r['log']['val_f1'].values.argmax()), best_val_acc=val_acc(r['log']).max())
                        for n, r in RUNS.items()]).sort_values('best_val_f1', ascending=False)
best = {m: summary[summary['model'] == m]['run'].iloc[0] for m in ['pmoe', 'mge'] if (summary['model'] == m).any()}
print('best on val:', best)
summary.round(4)

In [ ]:
# pMoE: every head setting
plotting.plot_comparison({label(r): r['log'] for r in runs_of('pmoe').values()})

In [ ]:
# MGE: every head setting
plotting.plot_comparison({label(r): r['log'] for r in runs_of('mge').values()})

In [ ]:
# head to head: the best pMoE vs the best MGE
plotting.plot_comparison({f'{label(RUNS[n])} (best {m})': RUNS[n]['log'] for m, n in best.items()})

## The two winners up close
pMoE: which patches each expert picks (needs the trained model, so not for a run read from its log).
MGE: how good each expert is on its own.

In [ ]:
n = best['pmoe']
net = RUNS[n]['net']
if net is None:
    print(f'{n} was read from its log: train it again in this session to see its patches')
else:
    images, _ = next(iter(val))
    top_idx, gates, grid = expert_choices(net, images[:64])
    print(f'{n}: top_idx {top_idx.shape} = (images, experts, l patches);  patch grid {grid}')
    for i in range(3):
        plotting.show_expert_patches(images[i].numpy(), top_idx[i], grid)

    # do the experts pick different patches? (the diagonal is l; big off-diagonal numbers = no specialisation)
    k = top_idx.shape[1]
    def overlap(i):
        return np.array([[len(set(top_idx[i, a]) & set(top_idx[i, b])) for b in range(k)] for a in range(k)])
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    sns.heatmap(overlap(0), annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax[0])
    ax[0].set(title='Shared patches, image 0', xlabel='expert', ylabel='expert')
    sns.heatmap(np.mean([overlap(i) for i in range(len(top_idx))], axis=0), annot=True, fmt='.1f', cmap='Blues',
                cbar=False, ax=ax[1])
    ax[1].set(title=f'Shared patches, mean of {len(top_idx)} images', xlabel='expert', ylabel='expert')
    plt.tight_layout(); plt.show()

In [ ]:
n = best['mge']
h = RUNS[n]['log']
outputs = ['ensemble', 'expert_0', 'expert_1', 'expert_2']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for o in outputs:
    ax[0].plot(h[f'val_{o}_accuracy'].values, marker='o', label=o)
    ax[1].plot(h[f'{o}_accuracy'].values, marker='o', label=o)
ax[0].set(title=f'{label(RUNS[n])}: validation accuracy per output', xlabel='epoch')
ax[1].set(title='training accuracy per output', xlabel='epoch')
ax[0].legend(); plt.tight_layout(); plt.show()
print('best val accuracy:', {o: round(float(h[f'val_{o}_accuracy'].max()), 4) for o in outputs})

## Test set (once, for the two winners)
The settings were chosen on validation; the test set only reports how good the chosen models are.

In [ ]:
y_test = np.concatenate([y.numpy() for _, y in test])
for m, n in best.items():
    net = RUNS[n]['net']
    if net is None:
        print(f'{m}: {n} was read from its log, no model to test (train it again in this session)')
        continue
    pred = net.predict(test, verbose=0)
    if isinstance(pred, (list, tuple)):          # MGE: the first output is the ensemble
        pred = pred[0]
    p = pred.argmax(1)
    print(f'{m:5s} {n}: test macro-F1 {f1_score(y_test, p, average="macro"):.4f}   accuracy {(p == y_test).mean():.4f}')

In [ ]:
# save the winners' weights (best epoch, because EarlyStopping restores the best weights)
for m, n in best.items():
    net = RUNS[n]['net']
    if net is None:
        continue
    path = f'{MODELS_DIR}/{n}.weights.h5'
    net.save_weights(path)
    print(f'saved {os.path.getsize(path) / 1e6:.1f} MB -> {path}')
    if USE_GITHUB:                               # Colab: the VM disk disappears, so commit it as well
        push_files({path: path}, f'model weights: {n}')